#  HaloBlocks Tutorial

Welcome to the official tutorial for **HaloBlocks**!

HaloBlocks is a modern, high-performance Python library for modular neural network components.
In this notebook we walk through **every major block family** with working examples so you can
hit the ground running.

**Table of Contents**
1. [Setup](#setup)
2. [The Three APIs](#apis)
3. [Attention Blocks](#attention)
4. [Positional Embeddings](#positional)
5. [MLP Block](#mlp)
6. [Mixture-of-Experts (MoE)](#moe)
7. [Transformer Block, Stacking, Operation Order & Decoder](#transformer)
8. [CompositeBlock — Building Pipelines](#composite)
9. [Mini GPT-style Language Model](#minigpt)
10. [Config-Driven Design](#config)
11. [Tips & Tricks](#tips)

<a id='setup'></a>
## 1.  Setup

Install the library and its only runtime dependencies (`torch`, `numpy`).

In [ ]:
# Uncomment to install
# !pip install haloblocks torch

import torch
import haloblocks as hb
from haloblocks import blocks

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device : {DEVICE}")
print(f"PyTorch      : {torch.__version__}")
print(f"HaloBlocks   : {hb.__version__ if hasattr(hb, '__version__') else '0.1.x'}")

<a id='apis'></a>
## 2.  The Three APIs

HaloBlocks offers three equivalent ways to create any block.
Pick whichever fits your workflow — they all produce identical `nn.Module` objects.

The string you pass to `blocks.<ClassName>` and to `hb.create('<ClassName>', ...)` / `{'type': '<ClassName>'}`
is the **registry key** — which is just the class name itself. For example,
multi-head self-attention is `MultiHeadAttention`; the separate single-head building block is
`HeadAttention` — use the key that matches the class you intend.

In [ ]:
# ── API 1: Keras-style direct access via ``from haloblocks import blocks`` ───
attn_a = blocks.MultiHeadAttention(emb_dim=256, num_heads=8)

# ── API 2: hb.create() with keyword arguments ───────────────────────────────
attn_b = hb.create('MultiHeadAttention', emb_dim=256, num_heads=8)

# ── API 3: hb.create() with a config dict (great for YAML/JSON) ─────────────
config = {'type': 'MultiHeadAttention', 'emb_dim': 256, 'num_heads': 8}
attn_c = hb.create(config)

print(type(attn_a), type(attn_b), type(attn_c))
print("All three are identical:", type(attn_a) == type(attn_b) == type(attn_c))

<a id='attention'></a>
## 3.  Attention Blocks

HaloBlocks ships a rich zoo of attention mechanisms. All blocks accept tensors of shape
`(batch, seq_len, emb_dim)` and return the same shape.

### 3.1 Single-Head Self-Attention

In [ ]:
# Basic single-head self-attention
sa = blocks.SelfAttention(emb_dim=128)

x = torch.randn(2, 16, 128)   # (batch=2, seq=16, dim=128)
out = sa(x)
print(f"Self-Attention  in: {x.shape}  →  out: {out.shape}")

# Return attention weights too
sa_w = blocks.SelfAttention(emb_dim=128, return_attn_weights=True)
out, weights = sa_w(x)
print(f"Attention weights shape: {weights.shape}")  # (batch, seq, seq)

### 3.2 Multi-Head Attention (MHA)

In [ ]:
# Standard MHA — emb_dim must be divisible by num_heads
mha = blocks.MultiHeadAttention(
    emb_dim=256,
    num_heads=8,
    drop_fact=0.1,
    causal_mask=True,   # auto-regressive / decoder style
)

x = torch.randn(4, 32, 256)   # (batch=4, seq=32, dim=256)
out = mha(x)
print(f"MHA  in: {x.shape}  →  out: {out.shape}")

# With QK-Norm (improves training stability)
mha_qk = blocks.MultiHeadAttention(emb_dim=256, num_heads=8, use_q_norm=True, use_k_norm=True)
print(f"MHA+QKNorm out: {mha_qk(x).shape}")

### 3.3 Grouped-Query Attention (GQA)

GQA reduces the KV-cache memory footprint by sharing key/value heads across groups
of query heads. Used in Llama 2/3, Mistral, Gemma, etc.

In [ ]:
# 8 query heads, 2 KV heads  →  4× KV-cache saving vs MHA
gqa = blocks.GroupedQueryAttention(
    emb_dim=512,
    num_heads=8,
    num_kv_heads=2,
    dropout=0.0,
)

x = torch.randn(2, 64, 512)
out = gqa(x)
print(f"GQA  in: {x.shape}  →  out: {out.shape}")

### 3.4 Cross-Attention

Cross-attention lets a query sequence attend to a separate context (e.g. encoder output
in a seq2seq model, or image tokens in a VLA model).

In [ ]:
ca = blocks.MultiHeadCrossAttention(emb_dim=256, num_heads=8)

query_seq   = torch.randn(2, 10, 256)   # decoder tokens
context_seq = torch.randn(2, 64, 256)   # encoder / image tokens

out = ca(query_seq, context_seq)
print(f"Cross-Attention  query: {query_seq.shape}, context: {context_seq.shape}  →  out: {out.shape}")

### 3.5 Sliding-Window Attention

Each token only attends to a local window of neighbours, giving O(n·w) complexity instead
of O(n²). Useful for long-context models.

In [ ]:
swa = blocks.SlidingWindowAttention(
    emb_dim=256,
    num_heads=8,
    window_size=16,   # each token sees ±8 neighbours
)

x = torch.randn(2, 128, 256)   # long sequence
out = swa(x)
print(f"SWA  in: {x.shape}  →  out: {out.shape}")

<a id='positional'></a>
## 4.  Positional Embeddings

HaloBlocks provides four drop-in positional encoding schemes.

### 4.1 Sinusoidal (fixed, no parameters)

In [ ]:
sin_pe = blocks.SinusoidalPositionalEmbedding(emb_dim=128, max_len=512)

x = torch.randn(2, 50, 128)   # (batch, seq, dim)
out = sin_pe(x)               # adds position signal to token embeddings
print(f"Sinusoidal PE  in: {x.shape}  →  out: {out.shape}")

### 4.2 Learned Embeddings (trainable)

In [ ]:
learned_pe = blocks.LearnedPositionalEmbedding(emb_dim=128, max_len=512)

out = learned_pe(x)
print(f"Learned PE  in: {x.shape}  →  out: {out.shape}")
print(f"Trainable parameters: {sum(p.numel() for p in learned_pe.parameters()):,}")

### 4.3 Rotary Positional Embedding (RoPE)

RoPE rotates Q and K vectors rather than adding to token embeddings directly.
Used in Llama, Falcon, Mistral, Gemma, and most modern LLMs.

In [ ]:
head_dim = 64
rope = blocks.RotaryPositionalEmbedding(head_dim=head_dim, max_len=1024)

# Simulate Q and K tensors from inside an attention head
# Shape: (batch, num_heads, seq_len, head_dim)
q = torch.randn(2, 8, 32, head_dim)
k = torch.randn(2, 8, 32, head_dim)

q_rot, k_rot = rope(q, k)
print(f"RoPE  q: {q.shape}  →  q_rot: {q_rot.shape}")
print(f"RoPE  k: {k.shape}  →  k_rot: {k_rot.shape}")

### 4.4 ALiBi (Attention with Linear Biases)

ALiBi adds a learnable linear bias to attention scores based on relative distance,
enabling length extrapolation without re-training.

In [ ]:
alibi = blocks.AlibiPositionalBias(num_heads=8)

# ALiBi returns a bias matrix to add to raw attention scores
seq_len = 32
bias = alibi(seq_len, device=torch.device('cpu'))
print(f"ALiBi bias shape: {bias.shape}")  # (num_heads, seq_len, seq_len)

<a id='MLP'></a>
## 5.  MLP Block

The `mlp` block is highly configurable — fully-connected layers with custom depth,
width, activation, bias, and optional last-layer activation.

In [ ]:
# A simple 3-layer MLP with GELU
mlp = blocks.MLP(
    input_dim=128,
    hidden_dims=[256, 256],
    output_dim=64,
    activation='gelu',
    bias=True,
    last_layer_activation=False,  # final layer is a linear projection
)

x = torch.randn(4, 128)
out = mlp(x)
print(f"MLP  in: {x.shape}  →  out: {out.shape}")

In [ ]:
# ReLU variant — no bias, last-layer activation
mlp_relu = blocks.MLP(
    input_dim=64,
    hidden_dims=[128],
    output_dim=64,
    activation='relu',
    bias=False,
    last_layer_activation=True,
)
x = torch.randn(8, 64)
print(f"MLP-ReLU  in: {x.shape}  →  out: {mlp_relu(x).shape}")

<a id='moe'></a>
## 6.  Mixture-of-Experts (MoE)

HaloBlocks implements the **DeepSeek MoE** architecture:
- **Shared experts** — always activated for every token.
- **Routed experts** — selected by a noisy Top-K router for load balancing.
- Each expert uses a **SwiGLU** activation internally.

In [ ]:
moe = blocks.DeepseekMoE(
    emb_dim=256,
    hid_dim=512,               # expert intermediate size
    num_router_exprts=8,       # total routed experts to choose from
    best_k=2,                  # number of routed experts activated per token
    num_shared_exprts=2,       # shared experts always active
)

x = torch.randn(2, 16, 256)   # (batch, seq, dim)
out = moe(x)
print(f"MoE  in: {x.shape}  →  out: {out.shape}")

total_params = sum(p.numel() for p in moe.parameters())
active_params = sum(p.numel() for p in moe.shared_experts.parameters())
# Each token activates 2/8 routed experts → roughly 25% of routed params
print(f"Total parameters  : {total_params:,}")
print(f"Always-on (shared): {active_params:,}")

<a id='transformer'></a>
## 7.  Transformer Block, Stacking, Operation Order & Decoder

### 7.1 Single Transformer Block

The `TransformerBlock` combines Pre-Norm MHA + residual with either a standard
MLP feed-forward or a full DeepSeek MoE layer.

In [ ]:
# Standard transformer block (MLP FFN)
tb = blocks.TransformerBlock(
    emb_dim=256,
    num_heads=8,
    mlp_dim=1024,
    drop_fact=0.1,
    causal_mask=False,
    use_moe=False,
)

x = torch.randn(4, 32, 256)
out = tb(x)
print(f"TransformerBlock (MLP)   in: {x.shape}  →  out: {out.shape}")

In [ ]:
# Transformer block with MoE FFN
tb_moe = blocks.TransformerBlock(
    emb_dim=256,
    num_heads=8,
    use_moe=True,
    moe_hid_scale=1.5,           # hid_dim = round(emb_dim * moe_hid_scale)
    moe_num_routed_experts=16,
    moe_top_k=4,
    moe_num_shared_experts=2,
    causal_mask=True,
)

out = tb_moe(x)
print(f"TransformerBlock (MoE)   in: {x.shape}  →  out: {out.shape}")

### 7.2 Stacked Decoder

`decoder_transformer` stacks N causal transformer blocks with a final LayerNorm.

In [ ]:
decoder = blocks.DecoderTransformer(
    num_layers=4,
    emb_dim=256,
    num_heads=8,
    use_moe=False,
    mlp_dim=512,
    drop_fact=0.0,
)

x = torch.randn(2, 32, 256)
out = decoder(x)
print(f"Decoder (4 layers)  in: {x.shape}  →  out: {out.shape}")

### 7.3 Composable Transformer Builder

Want more control? Use custom_transformer_block to plug any registered attention and FFN blocks into a transformer layer.

In [ ]:
# GQA Attention + DeepSeek MoE + RMSNorm
import torch
import haloblocks as hb
custom_tb = hb.create(
    'TransformerBlockBuilder',
    emb_dim=256,
    attn={'type': 'GroupedQueryAttention', 'num_heads': 8, 'num_kv_heads': 2},
    ffn={'type': 'DeepseekMoE', 'hid_dim': 512, 'num_router_exprts': 8, 'best_k': 2, 'num_shared_exprts': 1},
    norm='rmsnorm',
)

# x = torch.randn(2, 32, 256)
# out = custom_tb(x)
print(f"Custom Transformer block created.")

### 7.4 Stacking Layers

The builder gives you two ways to create a **multi-layer** transformer stack
(each layer gets its own independent weights plus a final normalisation layer).

**Option A — Programmatic:** call `.stack(num_layers)` on a builder instance.
**Option B — Config-driven:** use `StackedTransformerBlock` with a `num_layers` argument
(ideal for YAML / JSON configs and `CompositeBlock` pipelines).

In [ ]:
# ── Option A: programmatic stacking ──────────────────────────────────────────
# Build a single layer, then call .stack(N) to get a full N-layer stack.
single_layer = blocks.TransformerBlockBuilder(
    emb_dim=256,
    attn='GroupedQueryAttention',
    attn_kwargs={'num_heads': 8, 'num_kv_heads': 2},
    ffn={'type': 'DeepseekMoE', 'hid_dim': 512,
         'num_router_exprts': 8, 'best_k': 2, 'num_shared_exprts': 1},
    norm='rmsnorm',
    drop_fact=0.1,
)

stack_6 = single_layer.stack(num_layers=6)   # 6 layers + final RMSNorm

x = torch.randn(2, 32, 256)
out = stack_6(x)
print(f"Stacked (programmatic)  layers: 6  in: {x.shape}  →  out: {out.shape}")
print(f"  Parameters: {sum(p.numel() for p in stack_6.parameters()):,}")

In [ ]:
# ── Option B: config-driven stacking ─────────────────────────────────────────
# StackedTransformerBlock wraps TransformerBlockBuilder + .stack() into a single
# registry key — perfect for hb.create() and config dicts.

# Keyword style
stack_kw = hb.create(
    'StackedTransformerBlock',
    emb_dim=256,
    num_layers=4,
    attn='MultiHeadAttention',
    attn_kwargs={'num_heads': 8},
    norm='layernorm',
)

out = stack_kw(x)
print(f"StackedTransformerBlock (keyword)  layers: 4  in: {x.shape}  →  out: {out.shape}")

# Config-dict style — ideal for YAML / JSON experiment configs
stack_cfg = hb.create({
    'type': 'StackedTransformerBlock',
    'emb_dim': 256,
    'num_layers': 4,
    'attn': {'type': 'SlidingWindowAttention', 'num_heads': 8, 'window_size': 16},
    'ffn': {'type': 'MLP', 'input_dim': 256, 'hidden_dims': [1024], 'output_dim': 256},
    'norm': 'rmsnorm',
    'drop_fact': 0.05,
})

out = stack_cfg(x)
print(f"StackedTransformerBlock (config)   layers: 4  in: {x.shape}  →  out: {out.shape}")
print(f"  Parameters: {sum(p.numel() for p in stack_cfg.parameters()):,}")

### 7.5 Operation Order — Pre-Norm, Post-Norm & Cross-Attention

By default the builder uses **Pre-Norm** order (norm → compute → residual).
You can switch to **Post-Norm**, **Sandwich-Norm**, or add a **cross-attention**
sublayer simply by passing `operation_order`.

Each inner tuple is one *sublayer* wrapped by a residual connection.  Norms
before the compute op are pre-norms; norms after are post-norms.

| Preset | Order |
|---|---|
| `PRE_NORM` (default) | `(('norm','self_attn'), ('norm','ffn'))` |
| `POST_NORM` | `(('self_attn','norm'), ('ffn','norm'))` |
| `PRE_NORM_CROSS` | pre-norm with a cross-attention sublayer |
| `POST_NORM_CROSS` | post-norm with a cross-attention sublayer |
| `SANDWICH_NORM` | `(('norm','self_attn','norm'), ('norm','ffn','norm'))` |

In [ ]:
from haloblocks.core.builder import (
    PRE_NORM, POST_NORM, POST_NORM_CROSS, SANDWICH_NORM,
)

x   = torch.randn(2, 16, 256)
ctx = torch.randn(2, 32, 256)   # encoder / image tokens for cross-attention

# ── Post-Norm (original Transformer) ────────────────────────────────────────
post_norm_block = blocks.TransformerBlockBuilder(
    emb_dim=256,
    operation_order=POST_NORM,
)
out = post_norm_block(x)
print(f"POST_NORM          in: {x.shape}  →  out: {out.shape}")

# ── Post-Norm with cross-attention (encoder-decoder style) ──────────────────
cross_block = blocks.TransformerBlockBuilder(
    emb_dim=256,
    attn='GroupedQueryAttention',
    attn_kwargs={'num_heads': 8, 'num_kv_heads': 2},
    cross_attn='MultiHeadCrossAttention',
    cross_attn_kwargs={'num_heads': 8},
    norm='rmsnorm',
    operation_order=POST_NORM_CROSS,
)
out = cross_block(x, context=ctx)
print(f"POST_NORM_CROSS    in: {x.shape}  ctx: {ctx.shape}  →  out: {out.shape}")

# ── Sandwich-Norm (CogView-style) ──────────────────────────────────────────
sandwich_block = blocks.TransformerBlockBuilder(
    emb_dim=256,
    operation_order=SANDWICH_NORM,
)
out = sandwich_block(x)
print(f"SANDWICH_NORM      in: {x.shape}  →  out: {out.shape}")

In [ ]:
# A flat tuple is also accepted — auto-grouped by splitting at each compute op:
flat_block = blocks.TransformerBlockBuilder(
    emb_dim=256,
    operation_order=('self_attn', 'norm', 'cross_attn', 'norm', 'ffn', 'norm'),
)
out = flat_block(x, context=ctx)
print(f"Flat post-norm+cross  in: {x.shape}  ctx: {ctx.shape}  →  out: {out.shape}")

# You can also define a fully custom order — e.g. FFN before attention:
custom_order = blocks.TransformerBlockBuilder(
    emb_dim=256,
    operation_order=(('norm', 'ffn'), ('norm', 'self_attn')),
)
out = custom_order(x)
print(f"Custom (ffn-first)    in: {x.shape}  →  out: {out.shape}")

In [ ]:
# Stacking cross-attention layers — context flows through every layer
encoder_decoder = blocks.TransformerBlockBuilder(
    emb_dim=256,
    attn='MultiHeadAttention',
    attn_kwargs={'num_heads': 8},
    cross_attn='MultiHeadCrossAttention',
    cross_attn_kwargs={'num_heads': 8},
    norm='rmsnorm',
    drop_fact=0.1,
    operation_order=POST_NORM_CROSS,
).stack(num_layers=6)

out = encoder_decoder(x, context=ctx)
print(f"6-layer encoder-decoder  in: {x.shape}  ctx: {ctx.shape}  →  out: {out.shape}")
print(f"  Parameters: {sum(p.numel() for p in encoder_decoder.parameters()):,}")

# Same thing via config dict (for YAML / JSON configs)
enc_dec_cfg = hb.create({
    'type': 'StackedTransformerBlock',
    'emb_dim': 256,
    'num_layers': 6,
    'cross_attn': 'MultiHeadCrossAttention',
    'cross_attn_kwargs': {'num_heads': 8},
    'norm': 'rmsnorm',
    'operation_order': [['self_attn', 'norm'], ['cross_attn', 'norm'], ['ffn', 'norm']],
})
out = enc_dec_cfg(x, context=ctx)
print(f"Config-driven enc-dec    in: {x.shape}  ctx: {ctx.shape}  →  out: {out.shape}")

<a id='composite'></a>
## 8.  CompositeBlock — Building Pipelines

`CompositeBlock` chains blocks sequentially (like `nn.Sequential` but for HaloBlocks).
The output of block N is passed as input to block N+1.

In [ ]:
pipeline = hb.CompositeBlock([
    hb.create('SinusoidalPositionalEmbedding', emb_dim=256, max_len=512),
    hb.create('TransformerBlock',    emb_dim=256, num_heads=8, use_moe=False, mlp_dim=512),
    hb.create('TransformerBlock',    emb_dim=256, num_heads=8, use_moe=False, mlp_dim=512),
])

x = torch.randn(2, 32, 256)
out = pipeline(x)
print(f"Pipeline  in: {x.shape}  →  out: {out.shape}")

You can also define a composite model from a config dict:

In [ ]:
composite_config = {
    'type': 'CompositeBlock',
    'blocks': [
        {'type': 'MultiHeadAttention', 'emb_dim': 128, 'num_heads': 4},
        {'type': 'MLP', 'input_dim': 128, 'hidden_dims': [256, 128]},
    ]
}

model = hb.create(composite_config)
print(model)

<a id='minigpt'></a>
## 9.  Mini GPT-style Language Model

Let's put it all together and build a tiny GPT-like decoder-only language model
using HaloBlocks. This shows how every piece composes naturally with standard PyTorch.

In [ ]:
import torch.nn as nn

class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int = 8192,
        emb_dim: int    = 256,
        num_heads: int  = 8,
        num_layers: int = 4,
        max_seq_len: int = 512,
        mlp_dim: int    = 512,
        dropout: float  = 0.1,
    ):
        super().__init__()

        # Token + position embeddings
        self.tok_emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_emb = hb.create('LearnedPositionalEmbedding', emb_dim=emb_dim, max_len=max_seq_len)
        self.drop    = nn.Dropout(dropout)

        # Stack of causal transformer blocks
        self.blocks = nn.ModuleList([
            hb.create(
                'TransformerBlock',
                emb_dim=emb_dim,
                num_heads=num_heads,
                mlp_dim=mlp_dim,
                drop_fact=dropout,
                causal_mask=True,
                use_moe=False,
            )
            for _ in range(num_layers)
        ])

        self.ln_f  = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size, bias=False)

    def forward(self, token_ids):
        # token_ids: (batch, seq_len)
        x = self.tok_emb(token_ids)    # → (batch, seq_len, emb_dim)
        x = self.pos_emb(x)            # adds positional signal
        x = self.drop(x)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)       # → (batch, seq_len, vocab_size)
        return logits


model = MiniGPT().to(DEVICE)
total = sum(p.numel() for p in model.parameters())
print(f"MiniGPT parameters: {total/1e6:.2f} M")

# Forward pass
tokens = torch.randint(0, 8192, (2, 64)).to(DEVICE)
logits = model(tokens)
print(f"Logits shape: {logits.shape}")  # (2, 64, 8192)

Quick training step to verify gradients flow correctly:

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
for step in range(3):
    tokens = torch.randint(0, 8192, (4, 64)).to(DEVICE)
    targets = tokens.clone()

    logits = model(tokens)  # (B, T, V)
    loss   = criterion(logits.view(-1, 8192), targets.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Step {step+1}  loss: {loss.item():.4f}")

<a id='config'></a>
## 10.  Config-Driven Design

For reproducible experiments, define your whole model as a Python dict (or load from
JSON / YAML). This is useful for hyperparameter sweeps and logging configs to W&B / MLflow.

In [ ]:
# Imagine loading this from config.yaml
model_cfg = {
    'type': 'TransformerBlock',
    'emb_dim': 512,
    'num_heads': 16,
    'mlp_dim': 2048,
    'use_moe': True,
    'moe_num_routed_experts': 8,
    'moe_top_k': 2,
    'moe_num_shared_experts': 1,
    'causal_mask': True,
    'drop_fact': 0.05,
}

block = hb.create(model_cfg)
x = torch.randn(2, 32, 512)
print(f"Config block  in: {x.shape}  →  out: {block(x).shape}")

In [ ]:
import json

# Serialize for experiment tracking
print(json.dumps(model_cfg, indent=2))

<a id='tips'></a>
## 11.  Tips & Tricks

### Listing all available blocks

In [ ]:
from haloblocks.core.registry import BlockRegistry

all_blocks = sorted(BlockRegistry._registry.keys())
print(f"Registered blocks ({len(all_blocks)} total):")
for name in all_blocks:
    print(f"  {name}")

### Counting parameters

In [ ]:
def count_params(module, label=''):
    total     = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f"{label or type(module).__name__}")
    print(f"  Total      : {total:>12,}")
    print(f"  Trainable  : {trainable:>12,}")

count_params(blocks.TransformerBlock(emb_dim=512, num_heads=8, use_moe=False, mlp_dim=2048),
             'TransformerBlock (MLP, dim=512)')

### Moving to GPU

In [ ]:
block = blocks.MultiHeadAttention(emb_dim=256, num_heads=8).to(DEVICE)
x = torch.randn(2, 32, 256).to(DEVICE)
print(f"Running on: {next(block.parameters()).device}")

### Registering your own custom block

In [ ]:
import torch.nn as nn
from haloblocks.core.block import Block
from haloblocks.core.registry import BlockRegistry

@BlockRegistry.register('my_custom_block')
class MyCustomBlock(Block):
    def __init__(self, emb_dim=128, scale=1.0):
        super().__init__()
        self.proj  = nn.Linear(emb_dim, emb_dim)
        self.scale = scale

    def forward(self, x, **kwargs):
        return self.proj(x) * self.scale

# It's instantly available via all three APIs
custom = hb.create('my_custom_block', emb_dim=128, scale=0.5)
x = torch.randn(2, 16, 128)
print(f"Custom block out: {custom(x).shape}")

---

##  You're Ready!

You've covered the full HaloBlocks library:

-  Three usage APIs (blocks / create / config)
-  All attention variants (MHA, GQA, Cross, Sliding-Window…)
-  All positional encodings (Sinusoidal, Learned, RoPE, ALiBi)
-  MLP block
-  DeepSeek Mixture-of-Experts
-  TransformerBlock & DecoderTransformer
-  Stacked Transformers (`.stack()` & `StackedTransformerBlock`)
-  Operation order (Pre-Norm, Post-Norm, Sandwich, Cross-Attention)
-  CompositeBlock pipelines
-  A complete Mini-GPT
-  Custom block registration

**Next steps:**
- Browse the source under `src/haloblocks/blocks/` for every option each block accepts.
- Open an issue or PR on [GitHub](https://github.com/basaanithanaveenkumar/HaloBlocks) to contribute your own blocks!

Happy building! 